## Cell 1: Setup
**What this demonstrates**: Environment initialisation for Multi-Vector RAG — LangChain's `MultiVectorRetriever` and `InMemoryStore` for the two-layer architecture, OpenAI for embeddings, Anthropic for representation generation and answer synthesis.
**Expected output**: Setup confirmation with model names, representation types, and retrieval settings.

In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb anthropic openai python-dotenv

import os
import time
import pathlib
import uuid

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

ANSWER_MODEL  = 'claude-sonnet-4-6'
REP_MODEL     = 'claude-haiku-4-5-20251001'  # cheap model for representation generation
EMBED_MODEL   = 'text-embedding-3-small'
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 80
K_RETRIEVE    = 4   # representations retrieved; de-duped to unique source chunks
REP_TYPES     = ['summary', 'questions', 'keywords']  # representations per chunk

client        = Anthropic()
lc_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Answer model      : {ANSWER_MODEL}')
print(f'  Representation model: {REP_MODEL}')
print(f'  Embed model       : {EMBED_MODEL}')
print(f'  Chunk size        : {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap')
print(f'  Representations   : {REP_TYPES}')
print(f'  K retrieve        : {K_RETRIEVE} representations → unique source chunks')

## Cell 2: Data — Basel III Excerpt
**What this demonstrates**: Loading a structured regulatory document that attracts multiple query styles in practice. Compliance analysts ask thematic overview questions; risk managers ask specific ratio questions; operations teams search by exact regulatory terms. One embedding per chunk cannot serve all three — this is the setup problem for Multi-Vector RAG.
**Expected output**: Chunk count, word count, and concrete examples of the three query styles that will be tested.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

basel_text = (BASE_DIR / 'basel_iii_excerpt.txt').read_text(encoding='utf-8')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '.', ' '],
)
raw_chunks = [d.page_content for d in splitter.create_documents([basel_text])]

print(f'Loaded: {len(basel_text):,} characters, {len(raw_chunks)} chunks')
print()
print('The same document is queried three ways in practice:')
print()
print('  Compliance analyst (thematic):   "Summarize capital requirements"')
print('    -> best matched by: SUMMARY embedding (thematic language)')
print()
print('  Risk manager (question style):   "What is the leverage ratio?"')
print('    -> best matched by: QUESTIONS embedding (pre-generated Q&A)')
print()
print('  Operations (keyword lookup):     "Tier 1 capital"')
print('    -> best matched by: KEYWORDS embedding (domain term index)')
print()
print('One embedding per chunk misses two of three. Multi-Vector covers all.')
print()
print('Sample chunk (first):')
print('-' * 60)
print(raw_chunks[0][:400])

## Cell 3: Core — Representation Generation, Indexing, and Retrieval
**What this demonstrates**: Three functions generate representations (summary, questions, keywords) per chunk. `build_multi_vector_index` wires the two storage layers: a Chroma vector store holding all representations tagged with `doc_id`, and an `InMemoryStore` holding the original chunks. `multi_vector_rag` queries the vector store, surfaces matched rep types for inspection, then retrieves originals via `MultiVectorRetriever` for generation.
**Expected output**: Indexing log showing representation counts per chunk and total vectors indexed.

In [ ]:
def generate_summary(chunk: str) -> str:
    """Generate a 2-3 sentence thematic summary of a chunk.

    Matches overview and summarisation queries — language is thematic
    rather than verbatim, bridging the vocabulary gap for 'summarise X' queries.

    Args:
        chunk: Source text chunk.

    Returns:
        2-3 sentence summary string.
    """
    response = client.messages.create(
        model=REP_MODEL,
        max_tokens=200,
        messages=[{
            'role': 'user',
            'content': (
                'Summarise the following regulatory text in 2-3 sentences, '
                'capturing the main rule, threshold, or requirement.\n\n'
                f'{chunk}'
            ),
        }],
    )
    return response.content[0].text.strip()


def generate_questions(chunk: str) -> str:
    """Generate 3 questions that this chunk directly answers.

    Matches question-style queries — a user asking a question will embed
    close to pre-generated questions from the same chunk. This is the
    index-time analogue of HyDE (which does this at query time).

    Args:
        chunk: Source text chunk.

    Returns:
        Newline-separated question strings.
    """
    response = client.messages.create(
        model=REP_MODEL,
        max_tokens=200,
        messages=[{
            'role': 'user',
            'content': (
                'List exactly 3 questions that this regulatory text directly answers. '
                'Use only information present in the text. '
                'Format: one question per line, no numbering.\n\n'
                f'{chunk}'
            ),
        }],
    )
    return response.content[0].text.strip()


def extract_keywords(chunk: str) -> str:
    """Extract 5-7 key technical terms from a chunk.

    Matches exact-term lookup queries — 'Tier 1 capital' or 'leverage ratio'
    will embed close to a keyword document that lists those terms explicitly.

    Args:
        chunk: Source text chunk.

    Returns:
        Comma-separated keyword string prefixed with 'Keywords: '.
    """
    response = client.messages.create(
        model=REP_MODEL,
        max_tokens=100,
        messages=[{
            'role': 'user',
            'content': (
                'List 5-7 key technical terms, ratios, and regulatory concepts '
                'from this text, comma-separated. Include specific thresholds '
                'and acronyms where present.\n\n'
                f'{chunk}'
            ),
        }],
    )
    kw = response.content[0].text.strip()
    return f'Keywords: {kw}'


def build_multi_vector_index(
    chunks: list[str],
    collection_name: str = 'multi_vector_basel',
) -> tuple:
    """Build a Multi-Vector RAG index from a list of text chunks.

    For each chunk:
      1. Assign a unique doc_id (UUID).
      2. Generate summary, questions, and keywords representations.
      3. Tag each representation Document with {doc_id, rep_type}.
      4. Store all representations in Chroma (the search layer).
      5. Store original chunks in InMemoryStore (the retrieval layer).

    Args:
        chunks:          List of raw text chunks (level-0 source content).
        collection_name: Chroma collection name.

    Returns:
        Tuple of (retriever, vectorstore, doc_ids, all_rep_docs).
    """
    # Assign a stable UUID to each source chunk
    doc_ids = [str(uuid.uuid4()) for _ in chunks]

    all_rep_docs: list[Document] = []

    print(f'Generating representations for {len(chunks)} chunks...')
    for i, (chunk, doc_id) in enumerate(zip(chunks, doc_ids)):
        print(f'  Chunk {i + 1}/{len(chunks)}', end=' ', flush=True)

        summary   = generate_summary(chunk)
        questions = generate_questions(chunk)
        keywords  = extract_keywords(chunk)

        # Each rep Document carries doc_id so MultiVectorRetriever can link back
        all_rep_docs.extend([
            Document(page_content=summary,   metadata={'doc_id': doc_id, 'rep_type': 'summary'}),
            Document(page_content=questions, metadata={'doc_id': doc_id, 'rep_type': 'questions'}),
            Document(page_content=keywords,  metadata={'doc_id': doc_id, 'rep_type': 'keywords'}),
        ])
        print('done')

    print()
    print(f'Indexing {len(all_rep_docs)} representation vectors...', end=' ', flush=True)
    # Vector store holds only representations — this is the search layer
    vectorstore = Chroma.from_documents(
        all_rep_docs, lc_embeddings, collection_name=collection_name
    )
    print('done')

    # Document store holds originals — this is the retrieval layer
    # MultiVectorRetriever translates doc_id hits from vectorstore → originals here
    store = InMemoryStore()
    original_docs = [
        Document(page_content=c, metadata={'doc_id': did})
        for c, did in zip(chunks, doc_ids)
    ]
    store.mset(list(zip(doc_ids, original_docs)))

    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=store,
        id_key='doc_id',                 # metadata key that links reps to originals
        search_kwargs={'k': K_RETRIEVE}, # reps retrieved before de-duplication
    )

    return retriever, vectorstore, doc_ids, all_rep_docs


def multi_vector_rag(
    query: str,
    retriever: MultiVectorRetriever,
    vectorstore: Chroma,
) -> dict:
    """Query the Multi-Vector index and generate an answer.

    Two lookups happen in sequence:
      1. Direct vectorstore search → reveals which rep type the query matched
         (for inspection; these representations are never shown to the generator).
      2. MultiVectorRetriever.invoke() → translates matched doc_ids to original
         chunks and returns them for generation.

    Args:
        query:       User question.
        retriever:   MultiVectorRetriever built by build_multi_vector_index.
        vectorstore: Underlying Chroma store (used for direct rep inspection).

    Returns:
        dict with keys: query, rep_matches, original_chunks, answer, latency_ms.
    """
    t0 = time.perf_counter()

    # Step 1: direct vectorstore search — shows which representation was matched
    # This is for observability only; the generator never sees these
    rep_matches = vectorstore.similarity_search_with_score(query, k=K_RETRIEVE)

    # Step 2: MultiVectorRetriever translates rep doc_ids → original chunks
    original_chunks = retriever.invoke(query)

    # Build generation context from original chunks (not representations)
    context = '\n\n---\n\n'.join(
        f'[Chunk {i + 1}]\n{doc.page_content}'
        for i, doc in enumerate(original_chunks)
    )

    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=400,
        system=(
            'Answer using only the provided regulatory text. '
            'Cite specific ratios, thresholds, and article numbers where present.'
        ),
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}],
    )

    return {
        'query'           : query,
        'rep_matches'     : rep_matches,
        'original_chunks' : original_chunks,
        'answer'          : response.content[0].text.strip(),
        'latency_ms'      : (time.perf_counter() - t0) * 1000,
    }


# ── Build index ────────────────────────────────────────────────────────────────

retriever, vectorstore, doc_ids, all_rep_docs = build_multi_vector_index(raw_chunks)

rep_type_counts = {}
for d in all_rep_docs:
    rt = d.metadata['rep_type']
    rep_type_counts[rt] = rep_type_counts.get(rt, 0) + 1

print(f'\nIndex ready.')
print(f'  Source chunks in docstore  : {len(raw_chunks)}')
print(f'  Representation vectors     : {len(all_rep_docs)}')
print(f'  Vectors per type           : {rep_type_counts}')
print(f'  Storage multiplier         : {len(all_rep_docs) / len(raw_chunks):.1f}x chunks')

## Cell 4: Run — Three Query Styles, Same Index
**What this demonstrates**: Three queries — thematic summary, direct question, exact term — each phrased as a different user persona would phrase them. All three retrieve the correct original chunk via different matched representations.
**Expected output**: For each query: the top matched representation type, the original chunk returned, and the generated answer.

In [ ]:
QUERIES = [
    ('Summarize capital requirements',  'thematic — expect SUMMARY match'),
    ('What is the leverage ratio?',     'question — expect QUESTIONS match'),
    ('Tier 1 capital',                  'keyword — expect KEYWORDS match'),
]

results = []

for query, expected in QUERIES:
    print(f'Query : {query}')
    print(f'Style : {expected}')

    result = multi_vector_rag(query, retriever, vectorstore)
    results.append(result)

    # Show which representation type the query actually matched
    top_rep   = result['rep_matches'][0][0]
    top_score = result['rep_matches'][0][1]
    top_type  = top_rep.metadata.get('rep_type', 'unknown')
    top_doc_id = top_rep.metadata.get('doc_id', '')

    print(f'Matched rep type : {top_type.upper()} (score={top_score:.4f})')
    print(f'Matched rep text : {top_rep.page_content[:160].replace(chr(10), " ")}...')
    print()
    print(f'Original chunks returned : {len(result["original_chunks"])}')
    print(f'Top original chunk (200 chars):')
    top_original = result['original_chunks'][0].page_content if result['original_chunks'] else '(none)'
    print(f'  {top_original[:200].replace(chr(10), " ")}...')
    print()
    print(f'Latency : {result["latency_ms"]:.0f} ms')
    print()
    print('=' * 65)
    print('ANSWER')
    print('=' * 65)
    print(result['answer'])
    print()
    print('-' * 65)
    print()

## Cell 5: Inspect — Representation Match vs Original Chunk
**What this demonstrates**: The two-layer architecture in detail — for each query, showing the full matched representation (what was retrieved from the vector store) alongside the original chunk that was returned for generation. The representation that matched and the content the model generated from are always different objects.
**Expected output**: Per-query table of all matched representations with rep types and scores; the original chunk returned; and a storage breakdown.

In [ ]:
# ── Per-query: matched representations vs returned originals ──────────────────
for result, (query, expected) in zip(results, QUERIES):
    print(f'QUERY: "{query}"')
    print(f'Expected match: {expected}')
    print('=' * 65)

    # All matched representations from the vector store
    print('Matched representations (from vector store):')
    seen_types = []
    for doc, score in result['rep_matches']:
        rt = doc.metadata.get('rep_type', 'unknown')
        flag = ' ← TOP' if not seen_types else ''
        seen_types.append(rt)
        preview = doc.page_content[:120].replace('\n', ' ')
        print(f'  [{rt:10s} | score={score:.4f}]{flag}')
        print(f'    {preview}...' if len(doc.page_content) > 120 else f'    {preview}')

    print()

    # Original chunks returned to the generator
    print(f'Original chunks returned to generator: {len(result["original_chunks"])}')
    for i, orig in enumerate(result['original_chunks'][:2], 1):
        preview = orig.page_content[:200].replace('\n', ' ')
        print(f'  Chunk {i}: {preview}...' if len(orig.page_content) > 200 else f'  Chunk {i}: {preview}')

    print()
    # Key observation: the generator never saw the summary/questions/keywords
    top_type = result['rep_matches'][0][0].metadata.get('rep_type', '?')
    print(f'  ✦ Query matched a {top_type.upper()} representation')
    print(f'    but the generator received the ORIGINAL chunk — not the {top_type}')
    print()
    print('-' * 65)
    print()

# ── Storage breakdown ─────────────────────────────────────────────────────────
print('STORAGE BREAKDOWN')
print('=' * 65)
total_rep_chars  = sum(len(d.page_content) for d in all_rep_docs)
total_orig_chars = sum(len(c) for c in raw_chunks)
print(f'  Source chunks     : {len(raw_chunks)} docs, {total_orig_chars:,} chars (in docstore)')
print(f'  Representations   : {len(all_rep_docs)} vectors, {total_rep_chars:,} chars (in vectorstore)')
print(f'  Storage multiplier: {len(all_rep_docs) / len(raw_chunks):.1f}x chunks, '
      f'{total_rep_chars / total_orig_chars:.1f}x characters')
print()
for rt in REP_TYPES:
    rt_docs = [d for d in all_rep_docs if d.metadata['rep_type'] == rt]
    rt_chars = sum(len(d.page_content) for d in rt_docs)
    avg = rt_chars // len(rt_docs) if rt_docs else 0
    print(f'  {rt:12s}: {len(rt_docs)} vectors, avg {avg} chars/vector')

# ── Rep type match distribution across all 3 queries ─────────────────────────
print()
print('REPRESENTATION TYPE MATCHED (top hit per query):')
for result, (query, _) in zip(results, QUERIES):
    top_type = result['rep_matches'][0][0].metadata.get('rep_type', '?')
    top_score = result['rep_matches'][0][1]
    print(f'  "{query[:40]:40s}" → {top_type:10s} (score={top_score:.4f})')

## Cell 6: Fintech — Multi-Aspect Regulatory Search
**What this demonstrates**: Three regulatory analyst personas querying the same Basel III corpus using their natural language patterns. Each persona's query style maps to a different representation — without any routing logic. The same index, built once, serves all three.
**Expected output**: Per-persona query results with matched representation type, showing that diverse user populations are served by a single Multi-Vector index.

In [ ]:
ANALYST_QUERIES = [
    (
        'Risk analyst',
        'What are the consequences of failing to maintain the capital conservation buffer?',
        'question-style — should match QUESTIONS representation',
    ),
    (
        'Compliance officer',
        'Summarise the countercyclical buffer rules and when they apply',
        'thematic — should match SUMMARY representation',
    ),
    (
        'Operations / data team',
        'HQLA Level 2B haircut LCR',
        'keyword lookup — should match KEYWORDS representation',
    ),
]

print('FINTECH: MULTI-ASPECT REGULATORY SEARCH')
print('Use case: three analyst personas query the same Basel III index')
print('No routing logic — each persona\'s natural query style finds its match')
print()

for persona, query, expected in ANALYST_QUERIES:
    print(f'Persona : {persona}')
    print(f'Query   : {query}')
    print(f'Expected: {expected}')
    print()

    result = multi_vector_rag(query, retriever, vectorstore)

    top_rep   = result['rep_matches'][0][0]
    top_score = result['rep_matches'][0][1]
    top_type  = top_rep.metadata.get('rep_type', 'unknown')

    # Rep type distribution for this query
    type_hits = {}
    for doc, _ in result['rep_matches']:
        rt = doc.metadata.get('rep_type', 'unknown')
        type_hits[rt] = type_hits.get(rt, 0) + 1

    print(f'Top match: {top_type.upper()} (score={top_score:.4f})')
    print(f'All matched reps: { {k: v for k, v in type_hits.items()} }')
    print(f'Original chunks returned: {len(result["original_chunks"])}')
    print(f'Latency: {result["latency_ms"]:.0f} ms')
    print()
    print('Answer:')
    print(result['answer'])
    print()
    print('-' * 65)
    print()

print('MULTI-VECTOR RAG VALUE SUMMARY')
print('=' * 65)
print('  All three personas queried the same index built once at index time.')
print('  No query routing, no separate indexes, no query rewriting.')
print('  Each query found its natural match via a different representation.')
print()
print('  Flat index limitation: one embedding per chunk optimises for one')
print('  query style and degrades for the other two.')
print()
print(f'  Cost: {len(raw_chunks) * len(REP_TYPES)} LLM calls at index time '
      f'({len(raw_chunks)} chunks × {len(REP_TYPES)} reps).')
print(  '  Benefit: zero per-query overhead — retrieval is standard vector search.')